In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Phase 1: EDA & MSTL Baseline

1. Load the DataCo dataset
2. Turn daily transactions into weekly demand series
3. Clean the dataset by handling missing values and removing outliers
4. Exploratory Data Analysis (trends, seasonality, distribution)
5. MSTL forecast
6. MAE, RMSE, sMAPE

## Setup

In [12]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.config import DATA_RAW, DATAFILE
from src.utils.data_loader import DataLoader
from src.utils.data_cleaner import DataCleaner
from src.eda.exploratory_analysis import ExploratoryAnalysis
from src.forecasting.mstl_model import MSTLModel
from src.utils.visualization import (
    plot_time_series,
    plot_seasonality,
    plot_autocorrelation,
    plot_monthly_distribution,
    plot_forecast_comparison,
)

print('Setup complete.')

Setup complete.


## Load Data

In [13]:
loader = DataLoader(DATA_RAW / DATAFILE)
raw_df = loader.load()

summary = loader.get_summary()
print(f"Rows x Columns   : {summary['shape']}")
print(f"Date range       : {summary['date_range'][0].date()} - {summary['date_range'][1].date()}")
print(f"Duplicates dropped: {summary['duplicate_rows_dropped']}")
print()

# Show columns with missing values
missing = {k: v for k, v in summary['missing_values'].items() if v > 0}
print('Columns with missing values:')
for col, count in missing.items():
    print(f'  {col:<40} {count:>6} ({100*count/summary["shape"][0]:.1f}%)')

Rows x Columns   : (180519, 53)
Date range       : 2015-01-01 - 2018-01-31
Duplicates dropped: 0

Columns with missing values:
  Customer Lname                                8 (0.0%)
  Customer Zipcode                              3 (0.0%)
  Order Zipcode                            155679 (86.2%)
  Product Description                      180519 (100.0%)


## Clean Data

In [14]:
cleaner = DataCleaner(raw_df)

# each stage of cleaning
weekly_raw    = cleaner.aggregate_by_frequency()
weekly_filled = cleaner.handle_missing_values()
weekly_clean  = cleaner.remove_outliers()

print(f"Weekly periods   : {len(weekly_clean)}")
print(f"Demand range     : {weekly_clean.min():.0f} - {weekly_clean.max():.0f} units/week")
print(f"Mean / Std       : {weekly_clean.mean():.1f} / {weekly_clean.std():.1f}")
print()

# before and after
comparison = pd.DataFrame({
    'before': weekly_raw.describe().round(1),
    'after' : weekly_clean.describe().round(1),
})
print('before vs after:')
print(comparison.to_string())

Weekly periods   : 162
Demand range     : 205 - 2906 units/week
Mean / Std       : 2370.9 / 688.0

before vs after:
       before   after
count   162.0   162.0
mean   2370.9  2370.9
std     688.0   688.0
min     205.0   205.0
25%    2506.2  2506.2
50%    2609.5  2609.5
75%    2684.2  2684.2
max    2906.0  2906.0


## Train / Test Split

In [15]:
train, test = cleaner.train_test_split()

print(f"Train : {len(train)} weeks  ({train.index[0].date()} - {train.index[-1].date()})")
print(f"Test  : {len(test)} weeks  ({test.index[0].date()}  - {test.index[-1].date()})")
print()
print('Test set actuals:')
for date, val in test.items():
    print(f'  {date.date()}  -  {val:.0f} units')

Train : 134 weeks  (2015-01-04 - 2017-07-23)
Test  : 8 weeks  (2017-07-30  - 2017-09-17)

Test set actuals:
  2017-07-30  -  2409 units
  2017-08-06  -  2495 units
  2017-08-13  -  2462 units
  2017-08-20  -  2425 units
  2017-08-27  -  2572 units
  2017-09-03  -  2652 units
  2017-09-10  -  2469 units
  2017-09-17  -  2381 units


## EDA

In [16]:
eda = ExploratoryAnalysis(weekly_clean)
trend_info = eda.analyze_trends()

print(f"Trend direction  : {trend_info['direction']}")
print(f"Slope            : {trend_info['slope']:.2f} units/week")
print(f"R^2              : {trend_info['r_squared']:.3f}")
print(f"p-value          : {trend_info['p_value']:.4f}")
print(f"Total change     : {trend_info['pct_change']:+.1f}%  ({trend_info['first_value']:.0f} - {trend_info['last_value']:.0f})")
print(f"Stationary       : {trend_info['is_stationary']}")

fig = plot_time_series(weekly_clean, title='Full History (cleaned)')
fig.show()

Trend direction  : downward
Slope            : -8.29 units/week
R^2              : 0.320
p-value          : 0.0000
Total change     : -86.4%  (1511 - 205)
Stationary       : False


## EDA: Seasonality

In [17]:
season_info = eda.detect_seasonality(period=52)

print(f"Seasonal strength : {season_info['seasonal_strength']:.3f}")
print(f"ACF at lag 52     : {season_info['acf_at_period']:.3f}")
print(f"Seasonality detected: {season_info['has_seasonality']}")

fig_decomp = plot_seasonality(weekly_clean, period=52)
fig_decomp.show()

fig_acf = plot_autocorrelation(weekly_clean, lags=60)
fig_acf.show()

Seasonal strength : 0.013
ACF at lag 52     : -0.065
Seasonality detected: False


## EDA: Distribution

In [ ]:
dist_info = eda.get_distribution_stats()

print('Distribution:')
for key in ('mean', 'median', 'std', 'min', 'max', 'skewness', 'kurtosis', 'cv'):
    print(f'  {key:<12} {dist_info[key]:>10.3f}')
print(f'  {"outliers":<12} {dist_info["outlier_count"]:>10}')

print()
print('Monthly mean demand:')
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for m, mean_val in dist_info['monthly_means'].items():
    print(f'  {month_names[m-1]}  {mean_val:>7.0f}')

fig_monthly = plot_monthly_distribution(weekly_clean)
fig_monthly.show()

Distribution summary:
  mean           2370.858
  median         2609.500
  std             688.012
  min             205.000
  max            2906.000
  skewness         -2.345
  kurtosis          3.811
  cv                0.290
  outliers             19

Monthly mean demand:
  Jan     2096
  Feb     2465
  Mar     2631
  Apr     2657
  May     2576
  Jun     2532
  Jul     2594
  Aug     2579
  Sep     2606
  Oct     2021
  Nov     1995
  Dec     1829


## MSTL Forecast

In [19]:
model = MSTLModel(horizon=8, seasonal_periods=[52, 13])
model.fit(train)

forecast = model.predict()

print('Forecast vs Actual:')
print(f"  {'Date':<14} {'Forecast':>11} {'Actual':>9} {'Error':>10}")
print('  ' + '-' * 46)
for date, fc_val, act_val in zip(forecast.index, forecast.values, test.values):
    error = fc_val - act_val
    print(f'  {str(date.date()):<14} {fc_val:>10.1f} {act_val:>10.1f} {error:>+10.1f}')

Forecast vs Actual:
  Date              Forecast    Actual      Error
  ----------------------------------------------
  2017-07-30         2504.2     2409.0      +95.2
  2017-08-06         2646.6     2495.0     +151.6
  2017-08-13         2535.4     2462.0      +73.4
  2017-08-20         2634.1     2425.0     +209.1
  2017-08-27         2542.9     2572.0      -29.1
  2017-09-03         2587.3     2652.0      -64.7
  2017-09-10         2579.9     2469.0     +110.9
  2017-09-17         2765.3     2381.0     +384.3


## Evaluation

In [21]:
metrics = model.evaluate(test, forecast)

print('MSTL Model — Phase 1 Results')
print('=' * 35)
print(f"  MAE    : {metrics['mae']:.2f} units")
print(f"  RMSE   : {metrics['rmse']:.2f} units")
print(f"  sMAPE  : {metrics['smape']:.2f}%")
print()

fig_fc = plot_forecast_comparison(
    actual=test,
    forecast=forecast,
    title=f'MSTL Forecast vs Actual  |  MAE={metrics["mae"]:.0f}  sMAPE={metrics["smape"]:.1f}%',
    train=train,
)
fig_fc.show()

MSTL Model — Phase 1 Results
  MAE    : 139.77 units
  RMSE   : 175.41 units
  sMAPE  : 5.49%

